In [ ]:
import gymnasium as gym
import numpy as np
import torch
from torch import nn
from dqn_bonus import Trainer
import pickle 
import os

GAMMA = 0.99
TRUNC_LENGTH = 2000

In [2]:
# creating the modified environment
env = gym.make('MountainCar-v0', max_episode_steps=TRUNC_LENGTH)
seeds = [44, 63, 11, 15, 90, 21, 0, 12, 52, 8, 42, 32, 31, 10, 98]

In [ ]:
# Prioritised Experience Replay Analysis
replay_factors = [1, 4, 8]
per_res = {}

base_dir = "bonus_logs"  
os.makedirs(base_dir, exist_ok=True)

for rho in replay_factors:
    per_res_[rho] = []
    for s in seeds:
        env = gym.make('MountainCar-v0', max_episode_steps=TRUNC_LENGTH)

        trainer = Trainer(
            env=env,
            gamma=GAMMA,
            seed=s,
            use_wandb=True,
            wandb_project="RL_PA2",
            wandb_run_name=f"bonus_per_rho_{rho}_seed_{s}",
            trunc_length=TRUNC_LENGTH,
            use_per=True
        )

        returns = trainer.train(num_episodes=1500, replay_factor=rho)
        per_res_[rho].append(returns)

        save_dir = os.path.join(base_dir, f"per_rho_{rho}_seed_{s}")
        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, "train_rewards.npy"), returns)

        env.close()


In [ ]:
rho1_per = per_res[1]
rho4_per = per_res[4]
rho8_per = per_res[8]

In [5]:
def get_auc(rewards):
    return np.mean(rewards)

def mean_ci(x):
    x = np.array(x)
    mean = np.mean(x)
    std = np.std(x, ddof=1)   # sample std
    n = len(x)
    ci = 1.96 * std / np.sqrt(n)
    return mean, ci


def smooth(x, window=50):
    return np.convolve(x, np.ones(window)/window, mode='valid')

def mean_ci_smooth(arr, window=50):
    smoothed = np.array([smooth(r, window) for r in arr])
    
    mean = np.mean(smoothed, axis=0)
    std = np.std(smoothed, axis=0, ddof=1)
    n = smoothed.shape[0]
    
    ci = 1.96 * std / np.sqrt(n)
    return mean, ci



In [ ]:
rho1_per_vals = [get_auc(r) for r in rho1_per]
rho8_per_vals = [get_auc(r) for r in rho8_per]
rho4_per_vals = [get_auc(r) for r in rho4_per]

In [ ]:
rho1_mean, rho1_ci = mean_ci(rho1_per_vals)
rho8_mean, rho8_ci = mean_ci(rho8_per_vals)
rho4_mean, rho4_ci = mean_ci(rho4_per_vals)

print(f"ρ=1 PER: {rho1_mean:.2f} ± {rho1_ci:.2f}")
print(f"ρ=8 PER: {rho8_mean:.2f} ± {rho8_ci:.2f}")
print(f"ρ=4 PER: {rho4_mean:.2f} ± {rho4_ci:.2f}")

In [ ]:
with open("q4_results.pkl", "rb") as f:
    no_per_data = pickle.load(f)

rho1_no_per_vals = [get_auc(r) for r in no_per_data[1]]
rho8_no_per_vals = [get_auc(r) for r in no_per_data[8]]
rho4_no_per_vals = [get_auc(r) for r in no_per_data[4]]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PER Data
rho1_per_arr = np.array(rho1_per)
rho4_per_arr = np.array(rho4_per)
rho8_per_arr = np.array(rho8_per)

r1_mean, r1_ci = mean_ci_smooth(rho1_per_arr)
r4_mean, r4_ci = mean_ci_smooth(rho4_per_arr)
r8_mean, r8_ci = mean_ci_smooth(rho8_per_arr)

episodes = np.arange(len(r1_mean))

plt.figure(figsize=(9,6))

plt.plot(episodes, r1_mean, label="ρ=1")
plt.fill_between(episodes, r1_mean - r1_ci, r1_mean + r1_ci, alpha=0.2)

plt.plot(episodes, r4_mean, label="ρ=4")
plt.fill_between(episodes, r4_mean - r4_ci, r4_mean + r4_ci, alpha=0.2)

plt.plot(episodes, r8_mean, label="ρ=8")
plt.fill_between(episodes, r8_mean - r8_ci, r8_mean + r8_ci, alpha=0.2)

plt.xlabel("Episodes/Timesteps")
plt.ylabel("Aggregate Performance")
plt.title("PER (Window=50, 95% CI)")
plt.legend()
plt.grid()

plt.show()


# No PER Data
rho1_np_arr = np.array(no_per_data[1])
rho4_np_arr = np.array(no_per_data[4])
rho8_np_arr = np.array(no_per_data[8])

r1_mean_np, r1_ci_np = mean_ci_smooth(rho1_np_arr)
r4_mean_np, r4_ci_np = mean_ci_smooth(rho4_np_arr)
r8_mean_np, r8_ci_np = mean_ci_smooth(rho8_np_arr)

episodes = np.arange(len(r1_mean_np))


plt.figure(figsize=(9,6))

plt.plot(episodes, r1_mean_np, label="ρ=1")
plt.fill_between(episodes, r1_mean_np - r1_ci_np, r1_mean_np + r1_ci_np, alpha=0.2)

plt.plot(episodes, r4_mean_np, label="ρ=4")
plt.fill_between(episodes, r4_mean_np - r4_ci_np, r4_mean_np + r4_ci_np, alpha=0.2)

plt.plot(episodes, r8_mean_np, label="ρ=8")
plt.fill_between(episodes, r8_mean_np - r8_ci_np, r8_mean_np + r8_ci_np, alpha=0.2)

plt.xlabel("Episodes")
plt.ylabel("Aggregate Performance")
plt.title("Uniform Replay (Window=50, 95% CI)")
plt.legend()
plt.grid()

plt.show()

In [ ]:
rhos = [1, 4, 8]

# PER
means_per = []
cis_per = []

for vals in [rho1_per_vals, rho4_per_vals, rho8_per_vals]:
    m, c = mean_ci(vals)
    means_per.append(m)
    cis_per.append(c)

# NO PER
means_np = []
cis_np = []

for vals in [rho1_no_per_vals, rho4_no_per_vals, rho8_no_per_vals]:
    m, c = mean_ci(vals)
    means_np.append(m)
    cis_np.append(c)

In [ ]:
plt.figure(figsize=(8,5))

#No PER
plt.errorbar(rhos, means_np, yerr=cis_np,
             fmt='o-', capsize=5, label="No PER", color = "red")
#PER
plt.errorbar(rhos, means_per, yerr=cis_per,
             fmt='o-', capsize=5, label="PER", color = "blue")


plt.xlabel("Replay Factor")
plt.ylabel("Aggregate Performance (AUC)")
plt.title("Aggregate Performance vs Replay Factor (95% CI)")
plt.xticks([1, 4, 8])
plt.legend()
plt.grid()

plt.show()

In [ ]:
print("Replay Factor | PER (mean ± CI) | No PER (mean ± CI)")
print("-----------------------------------------------------")

for i, rho in enumerate(rhos):
    print(f"{rho:^13} | "
          f"{means_per[i]:.2f} ± {cis_per[i]:.2f} | "
          f"{means_np[i]:.2f} ± {cis_np[i]:.2f}")